In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import seaborn as sns
import sys

sys.path.append('../../processors')
from json_parser import flatten_data # type: ignore

pd.set_option('display.max_columns', None)

In [ ]:
json_file = '../../data/raw/AN2023.json' 

df = pd.read_json(json_file)
df = flatten_data(json_file)
# Might also want to extract descrizione_impianto for future use

print(df.shape) # Rows, Columns


In [ ]:
df['edificio_id'] = df.index + 1
df.head()

In [ ]:
print(df['servizio_0_simulato'].value_counts())

## Missing values

In [ ]:
# features with missing values
features_na = [features for features in df.columns if df[features].isnull().sum() > 1]

# feature name + percentage of missing values
for feature in features_na:
    print(feature, np.round(df[feature].isnull().mean(), 5), '% missing values')

### Relationship between missing values and the target variable (EPnren)

In [ ]:

for feature in features_na:
    data = df.copy()
    
    # 1 to indicate missing, 0 NOT missing
    data[feature] = np.where(data[feature].isnull(), 1, 0)
    
    # calculate the mean of the target for both groups
    data.groupby(feature)['epglnren'].median().plot.bar()
    plt.title(feature)
    plt.show()

## Numerical values

In [ ]:
numerical_features = df.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features))

df[numerical_features].head()

## Temporal values

In [ ]:
time_features = [feature for feature in df if 'sopralluogo' in feature or 'validita' in feature]
time_features

In [ ]:
# exploring the content of these features

for feature in time_features:
    print(feature, df[feature].unique())  # did not consider anno_costruzione because it is a numerical predictor, not a time feature

### Is there any relation between anno_costruzione and epglnren?

In [ ]:
df.groupby('anno_costruzione')['epglnren'].median().plot.bar(figsize=(12, 6))
plt.xlabel('anno_costruzione')
plt.ylabel('Median epglnren')
plt.title('Median epglnren by anno_costruzione')
plt.xticks(rotation=70, ha='right', fontsize=5)
plt.tight_layout()
plt.show()

##### no further data exploration required for the temporal variables, since they have no relation to the prediction output

## Continuous and Discrete variables

In [ ]:
discrete_features = [feature for feature in df if len(df[feature].unique()) < 5 and feature not in time_features]
print('Discrete features count: ', len(discrete_features))

In [ ]:
discrete_features

In [ ]:
# exploring the relationship between the discrete features and the target variable epglnren
for feature in discrete_features:
    try:
        grouped = df.groupby(feature)['epglnren'].median()
        # remove NaN medians and empty groups
        grouped = grouped.dropna()
        if grouped.empty:
            print(f"Skipping '{feature}': no groups or all NaN medians (unique count: {df[feature].nunique(dropna=True)})")
            continue
        plt.figure(figsize=(6, 4))
        grouped.plot.bar()
        plt.xlabel(feature)
        plt.ylabel('epglnren')
        plt.title(feature)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error plotting feature '{feature}': {e}")
        # diagnostic information
        print('dtype:', df[feature].dtype, 'non-null:', df[feature].notnull().sum(), 'unique:', df[feature].nunique(dropna=True))
        try:
            display(df[[feature, 'epglnren']].head())
        except Exception:
            pass

In [ ]:
# unstructured mixed text features (most likely unwanted and not useful for the prediction task)
unstructured_text_features = ['software_utilizzato', 'cap', 'comune', 'codice_istat_comune', 'informazioni_aggiuntive', 'informazioni_miglioramento', 'piano', 'altra_motivazione', 'provincia', 'comune']
print('Unstructured text features count: ', len(unstructured_text_features))

print('Unstructured text features: ', unstructured_text_features)

In [ ]:
# Continuous features
continuous_features = [feature for feature in df if feature not in discrete_features + time_features + unstructured_text_features and feature != 'classe_energetica' and feature != 'classe_energetica_raggiungibile']
print('Continuous features count: ', len(continuous_features))

In [ ]:
# histograms for continuous features
for feature in continuous_features:
    data = df.copy()
    data[feature].hist(bins=30)
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.title(feature)
    plt.show()

In [ ]:
## logarithmic transformation for skewed features
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data['epglnren'] = np.log(data['epglnren'])
        plt.scatter(data[feature], data['epglnren'])
        plt.xlabel(feature)
        plt.ylabel('epglnren')
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

## Outliers

In [ ]:
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data.boxplot(column=feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

In [ ]:
## Categorical features
categorical_features = ['classe_energetica', 'classe_energetica_raggiungibile', 'zona_climatica']
categorical_features

In [ ]:
df[categorical_features].head()

In [ ]:
for feature in categorical_features:
    print(f'feature: {feature}, number of categories: {len(df[feature].unique())}')

In [ ]:
## relationship between categorical features and the target variable epglnren
for feature in categorical_features:
    df.groupby(feature)['epglnren'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('epglnren')
    plt.title(feature)
    plt.show()

## Correlation matrix

In [ ]:
corr = df[numerical_features].corr()
plt.figure(figsize=(50, 30))
mask = corr.isnull()
# copy where NaNs are a distinct value just for coloring
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5,
            cbar_kws={'label': 'Pearson r'})
# overlay text at NaN locations:
for i, j in zip(*np.where(mask)):
    plt.text(j+0.5, i+0.5, 'NA', ha='center', va='center', color='black', fontsize=6)
plt.show()

### Structural issue with the servizio_* features, these features are very sparse and conditional, with semantic inconsistency 

In [ ]:
# The servizio_* features are very sparse and conditional, with semantic inconsistency. We will reshape them into a long format for better analysis.

servizi = [] # [1: invernale, 2: estiva, 3: acs, 4: impianti combinati, 5: fonti rinnovabili, 6: vent. meccanica, 7: illuminazione, 8: trasporto persone o cose]

for i in range(0, 8):  # servizio_0 to servizio_7 ^^^^^^
    cols = {
        "simulato": f"servizio_{i}_simulato",
        "epnren": f"servizio_{i}_epnren",
        "epren": f"servizio_{i}_epren",
        "efficienza": f"servizio_{i}_efficienza",
        "potenza_nominale": f"servizio_{i}_pnominale"
    }
    
    existing = [c for c in cols.values() if c in df.columns]

    if len(existing) < 2:  # If less than 2 of the expected
        continue           # Skip this servizio index
    
    temp = df[['edificio_id'] + existing].copy()

    rename_map = {
        cols["simulato"]: "simulato",
        cols["epnren"]: "epnren",
        cols["epren"]: "epren",
        cols["efficienza"]: "efficienza",
        cols["potenza_nominale"]: "potenza_nominale"
    }

    temp = temp.rename(columns = {k: v for k, v in rename_map.items() if k in temp.columns})
    temp["servizio_idx"] = i
    
    servizi.append(temp)

long_df = pd.concat(servizi, ignore_index=True)
long_df = long_df.dropna(subset=["epnren", 'efficienza'], how = "all") # Drop rows where the system doesn't exist
counts = long_df['servizio_idx'].value_counts().sort_index()
print(counts)

In [ ]:
long_df.head() # edificio_id is important here

### Extract information and useful consistent features, then drop all raw servizio_* features

In [ ]:
# Features related to efficiency
efficiency_stats = long_df.groupby('edificio_id')['efficienza'].agg(
    efficienza_media = 'mean'
)

# Servizi counts
num_servizi = long_df.groupby('edificio_id').size().rename('num_servizi')

# Real vs Simulato

# real_df = long_df[long_df['simulato'] == 0]
# sim_df = long_df[long_df['simulato'] == 1]
# real_efficienza = real_df.groupby('edificio_id')['efficienza'].mean().rename('efficienza_media_reale')
# sim_efficienza = sim_df.groupby('edificio_id')['efficienza'].mean().rename('efficienza_media_simulata')

# Simulation features

sim_stats = long_df.groupby('edificio_id')['simulato'].agg(
     num_simulati = 'sum',
 )

# Potenza nominale stats
potenza_stats = long_df.groupby('edificio_id')['potenza_nominale'].agg(
    potenza_tot = 'sum'
)

In [ ]:

# Merge everything back together
df_new = df.copy()

for feature in [num_servizi, efficiency_stats, potenza_stats, sim_stats]:
    df_new = df_new.merge(feature, on='edificio_id', how='left')

servizio_cols = [c for c in df.columns if c.startswith('servizio_')]
df_new = df_new.drop(columns=servizio_cols)


In [ ]:
df_new.shape

In [ ]:
df_new.head()

In [ ]:
columns_to_drop = [
    'epglren',
    'co2',
    'ephnd',
    'software_utilizzato',
    'ephndlim',
    'epglnrenrs',
    'data_sopralluogo',
    'inizio_validita',
    'fine_validita',
    'classe_energetica',
    'motivazione',
    'informazioni_miglioramento',
    'inverno',
    'estate',
    'classe_energetica_raggiungibile',
    'epglnren_raggiungibile',
    'altra_motivazione',
    'informazioni_aggiuntive',
    'provincia',
    'comune',
    'codice_istat_comune',
    'dpr412',
    'oggetto_attestato',
    'numero_unita',
    'proprieta_edificio',
    'cap',
    'edificio_id',
    'energiaesportata',
    'vetenergiaesportata',
    'nzeb', # dont know what nzeb is, it is boolean, but around 90% of the values are False, and it is not clear what it means, so we will drop it for now
    'gradi_giorno', # while it could be important in other countries, it isnt in  Italy, besides we already have zona_climatica which is more relevant
    'presenza_illuminazione', # in 99% of cases, this does not impact total energy consumption
    'presenza_trasporto_pc', # 99% of properties have this as False, and it is not clear how it would impact energy consumption
    'presenza_vent_meccanica' # 99% of properties have this as False, this would only be relevant for non residential buildings, we are focusing on residential builddings
    ]

## Correlation heatmap for new_df before dropping unwanted columns

In [ ]:
numerical_features_new = df_new.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features_new))

corr_new = df_new[numerical_features_new].corr()
plt.figure(figsize=(25, 15))
mask = corr_new.isnull()
# copy where NaNs are a distinct value just for coloring
sns.heatmap(corr_new, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5,
            cbar_kws={'label': 'Pearson r'})
# overlay text at NaN locations:
for i, j in zip(*np.where(mask)):
    plt.text(j+0.5, i+0.5, 'NA', ha='center', va='center', color='black', fontsize=6)
plt.show()

## Correlation heatmap for new_df after dropping unwanted columns

In [ ]:
df_pure = df_new.drop(columns=columns_to_drop)
df_pure.shape

In [ ]:
numerical_features_pure = df_pure.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features_pure))

corr_pure = df_pure[numerical_features_pure].corr()
plt.figure(figsize=(15, 9))
mask = corr_pure.isnull()
# copy where NaNs are a distinct value just for coloring
sns.heatmap(corr_pure, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5,
            cbar_kws={'label': 'Pearson r'})
# overlay text at NaN locations:
for i, j in zip(*np.where(mask)):
    plt.text(j+0.5, i+0.5, 'NA', ha='center', va='center', color='black', fontsize=6)
plt.show()

### Domain concepts do not recommend blindly dropping the high correlation cluster of sup/vol_* due to the significance of physical importance and signal they carry.
I have visualized the initial dataset and ran tests to understand what kind of data we have, and came to the conclusion that the data was too messy and too badly put together for me to extract any valuable information, now that the data has been cleaned successfully, let's properly examine these features.

In [ ]:
df_pure.info()

### Some of these features are categorical (nominal/ordinal) and not numerical, encoding is required for these features
these features include:
anno_costruzione, piano, destinazione_uso, tipologia_edilizia and tipologia_costruttiva, zona_climatica

In [ ]:
# lets start by categorizing anno_costruzione and other features, and then correcting dtypes
df_pure['anno_costruzione'] = pd.cut(
    df_pure['anno_costruzione'],
    bins = [df['anno_costruzione'].min() - 1, 1930, 1945, 1960, 1975, 1985, 1991, 2005, 2007, df['anno_costruzione'].max() + 1],
    labels = ['before 1930', '1930-1945', '1946-1960', '1961-1975', '1976-1985', '1986-1991', '1992-2005', '2006-2007', 'after 2007'], right=False
    )

In [ ]:
df_pure.rename(columns = {
    'destinazione_uso': 'd_uso_Residenziale'}, inplace=True)
df_pure['d_uso_Residenziale'] = ~df_pure['d_uso_Residenziale'].astype('bool')
df_pure['d_uso_Residenziale'].value_counts()

In [ ]:
df_pure['tipologia_edilizia'] = df_pure['tipologia_edilizia'].replace({
    0: 'monofamiliare',
    1: 'bifamiliare',
    2: 'plurifamiliare',
    3: 'in linea',
    4: 'schiera',
    5: 'corte',
    6: 'torre',
    7: 'blocco',
    8: 'piastra',
    9: 'altro'
}).astype('category')

In [ ]:
df_pure['tipologia_costruttiva'] = df_pure['tipologia_costruttiva'].replace({
    0: 'muratura portante',
    1: 'c.a. con laterizi',
    2: 'c.a. con chiusure continue in vetro',
    3: 'c.a. con chiusure in pannelli prefabbricati',
    4: 'acciaio con chiusure in muratura',
    5: 'acciaio con chiusure continue in vetro',
    6: 'acciaio con chiusure in pannelli prefabbricati',
    7: 'legno',
    8: 'prefabbricata in c.a.',
    9: 'mista (c.a. + laterizi)',
    10: 'mista (c.a. + acciaio)',
    11: 'mista (acciaio + muratura)',
    12: 'mista (muratura + legno)',
    13: 'mista (altro)',
    14: 'altro'
}).astype('category')

In [ ]:
df_pure['zona_climatica'] = df_pure['zona_climatica'].astype('category')

In [ ]:
df_pure['num_simulati'] = df_pure['num_simulati'].astype(int)

### One last categorical feature remains, i have left it for last since its the messiest potentially useful column in this dataset
and that feature is 'piano'

In [ ]:
df_pure['piano'].value_counts()

In [ ]:
from piano_cleaner import clean_piano

df_pure['piano_clean'] = clean_piano(df_pure['piano'])

In [ ]:
df_pure = df_pure.drop(columns = ['piano'])
df_pure = df_pure.rename(columns= {'piano_clean': 'piano'})

In [ ]:
df_pure.info()

### The dataframe is now completely clean
now we re-explore the data, the outputs should be much better

### Missing values

In [ ]:
features_na = [features for features in df_pure.columns if df_pure[features].isnull().sum() > 1]

if features_na == []:
    print('No missing values')
else:
    for feature in features_na:
        print(feature, np.round(df_pure[feature].isnull().mean(), 5), '% missing values')

### Numerical values

In [ ]:
numerical_features = df_pure.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features))

df_pure[numerical_features].head()

In [ ]:
## histograms for numerical features

for feature in numerical_features:
    data = df_pure.copy()
#   data[feature] = np.log1p(data[feature])
    data[feature].hist(bins=30)
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.title(feature)
    plt.show()

It appears most of our numerical values are skewed.

### Categorical features

In [ ]:
categorical_features = ['anno_costruzione', 'zona_climatica', 'tipologia_edilizia', 'tipologia_costruttiva', 'piano']
df_pure[categorical_features].head()

### How do the categorical features correlate with epglnren?

In [ ]:
for feature in categorical_features:
        
    df_pure.groupby(feature)['epglnren'].median().plot.bar(figsize=(12, 6))
    plt.xlabel(feature)
    plt.ylabel('Median epglnren')
    plt.title(f'Median epglnren by {feature}')
    if feature != 'tipologia_costruttiva':
        plt.xticks(rotation=0, ha='center', fontsize=10)
    else:
        plt.xticks(rotation=70, ha='right', fontsize=7)
    plt.tight_layout()
    plt.show()

### Outliers

In [ ]:
for feature in numerical_features:
    data = df_pure.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    data.boxplot(column=feature)
    plt.ylabel(feature)
    plt.title(feature)
    plt.show()